[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/C.%20Core%20Yard%20%26%20Land-Side%20Operations%20%28The%20Heart%20of%20the%20Terminal%29/25.%20The%20Dynamic%20Truck%20Appointment%20System%20Problem/P25-Tier-4_executed.ipynb)

---


# 25. The Dynamic Truck Appointment System Problem
## Tier 4: Q-Learning for Appointment Management

### Goal

This notebook builds a small, stochastic simulation of truck appointment management and trains a tabular Q-Learning agent to choose actions in that simulation. The agent observes a simplified system state, selects an action, receives a scalar reward, and updates its estimates of future returns. The notebook then evaluates the learned policy and visualizes the recorded training and evaluation data.

The purpose is to study the Q-Learning solution method and its behavior in a compact example. The simulation is illustrative: it is not a calibrated terminal model, and its results should not be treated as operational recommendations.

### Inputs and outputs

The inputs are the environment parameters, the Q-Learning parameters, and random events generated by the simulation. The main outputs are the learned Q-table, episode-level training records, policy evaluation statistics, and visualizations of those records. Evaluation statistics are produced when the notebook is run; they are not specified in advance.

### Model assumptions and scope

- The environment contains a fixed number of trucks and appointment slots. Delays, slot utilization, congestion, time period, and action outcomes are represented in simplified form.
- A state key records which trucks have delays greater than 30 minutes, a bucket for average slot utilization, the time-period category, and the congestion category. This aggregation keeps a tabular representation manageable, but distinct detailed situations can map to the same key.
- Rescheduling updates the truck's assigned slot and a utilization proxy. Capacity adjustment also changes that proxy; the simulation does not represent physical yard capacity or a detailed service process.
- An incentive has a stochastic reward benefit for eligible trucks. The simulation does not model a truck's appointment changing after accepting an incentive.
- Episodes are fixed-length training segments in a continuing environment. The environment is not reset between segments, and the transition function does not signal a terminal state.
- The reward is a simplified scalar score combining action effects, waiting-cost penalties, operating-cost penalties, and a congestion penalty. Its components and units are illustrative rather than calibrated financial estimates.
- There is no historical terminal dataset or measured baseline-method result supplied here. Consequently, the notebook does not claim real-world savings, waiting-time reductions, convergence, or superiority over another method.

### The Q-Learning model

A **state** is the information available to the agent when it chooses an action. An **action** is one of the scheduling or capacity decisions available in that state. A **reward** is a scalar feedback signal: higher values are preferred, and costs are represented as negative contributions. Q-Learning estimates the long-run value of taking action $a$ in state $s$ and then following its policy.

Let $S$ be the set of encoded states, $A$ the set of actions, $r$ the reward, and $s'$ the next state. The tabular Q-Learning update is

$$
Q_{t+1}(s,a) = Q_t(s,a) + \alpha\left[r_{t+1} + \gamma \max_{a' \in A} Q_t(s',a') - Q_t(s,a)\right].
$$

Here, $Q_t(s,a)$ is the current estimate, $\alpha$ is the learning rate, and $\gamma$ is the discount factor. Because this environment is continuing, the update includes the estimated value of the next state. The implementation uses a zero-initialized value for action-state pairs not yet recorded in the Q-table.

The agent uses an $\epsilon$-greedy action rule. With probability $\epsilon$, it explores by selecting an action at random; otherwise, it selects an action with the largest current Q-value. Exploration makes it possible to gather information about actions that have not yet been tried. The exploration probability decays across training episodes, subject to a minimum value.

A tabular method stores a value for each encountered state-action pair. Its storage and learning requirements grow with the number of distinct encoded states and available actions. State aggregation reduces the size of this example, but it also discards information; Q-Learning does not guarantee that this particular simulation or training run finds a globally optimal operational policy.

### Learning sequence

1. Represent truck and system information as state objects, then map each state to a hashable key for the Q-table.
2. Define the available actions and the environment's reward and transition behavior.
3. Select actions with an $\epsilon$-greedy policy and update Q-values using the Bellman-style learning rule above.
4. Train over fixed-length episode segments while recording rewards and exploration rates.
5. Evaluate the learned, non-exploratory policy in the same stochastic environment and inspect the resulting diagnostics.

### Reading the results

Training rewards are observations from this stochastic simulation, not proof of convergence or evidence of real-world performance. Compare the episode reward series with its moving average, inspect the exploration schedule, and review the evaluation reward distribution and action counts. The Q-value heatmap shows only a sample of recorded states and actions. A quantitative comparison with another scheduling method requires measurements from that method under comparable scenarios and a shared evaluation protocol.


In [ ]:
from collections import defaultdict
from copy import deepcopy
from dataclasses import dataclass
import random
import time
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
@dataclass
class QLearningParameters:
    """Parameters controlling tabular Q-Learning."""

    learning_rate: float = 0.1
    discount_factor: float = 0.95
    exploration_rate: float = 1.0
    exploration_decay: float = 0.995
    min_exploration_rate: float = 0.01
    num_episodes: int = 10000
    max_steps_per_episode: int = 100


@dataclass
class TruckState:
    """Simplified appointment and delay information for one truck."""

    id: int
    current_slot: int
    expected_delay: int  # minutes
    urgency_level: int  # 1 = low, 2 = medium, 3 = high
    location: str  # on_time, delayed, or very_delayed


@dataclass
class SystemState:
    """Snapshot of the information used to describe the system."""

    truck_states: List[TruckState]
    slot_utilization: Dict[int, int]  # slot ID -> utilization percentage
    current_time_period: int  # 1 = peak, 2 = off-peak
    congestion_level: int  # 1 = low, 2 = medium, 3 = high

    def get_state_key(self) -> Tuple:
        """Return a compact, hashable state representation for the Q-table."""
        delay_pattern = tuple(
            truck.expected_delay > 30 for truck in self.truck_states
        )
        average_utilization = float(np.mean(list(self.slot_utilization.values())))
        # Utilization is a percentage, so cap the bucket at 3 for values up to 100%.
        utilization_bucket = min(3, int(average_utilization / 25))

        return (
            delay_pattern,
            utilization_bucket,
            self.current_time_period,
            self.congestion_level,
        )


@dataclass
class Action:
    """A scheduling or capacity decision available to the agent."""

    type: str  # maintain, reschedule, offer_incentive, or adjust_capacity
    truck_id: Optional[int] = None
    target_slot: Optional[int] = None
    incentive_amount: Optional[float] = None
    capacity_delta: Optional[int] = None


@dataclass
class TruckAppointmentEnvironment:
    """Stochastic, illustrative environment for appointment decisions."""

    num_trucks: int = 4
    num_slots: int = 6
    slot_capacity: int = 2
    waiting_cost_per_minute: float = 5.0
    rescheduling_cost: float = 50.0
    incentive_cost_per_unit: float = 25.0
    operating_cost_per_slot: float = 100.0

    def __post_init__(self) -> None:
        if self.num_trucks <= 0 or self.num_slots <= 0:
            raise ValueError("The number of trucks and slots must be positive.")

        self.trucks: List[TruckState] = []
        for index in range(self.num_trucks):
            delay = random.choice([0, 15, 30, 45, 60])
            urgency = 1 if delay < 20 else (2 if delay < 40 else 3)
            location = (
                "on_time" if delay == 0
                else ("delayed" if delay < 45 else "very_delayed")
            )
            self.trucks.append(
                TruckState(
                    id=index + 1,
                    current_slot=(index % self.num_slots) + 1,
                    expected_delay=delay,
                    urgency_level=urgency,
                    location=location,
                )
            )

        self.slot_utilization = {
            slot_id: random.randint(40, 90)
            for slot_id in range(1, self.num_slots + 1)
        }
        self.actions = self._define_action_space()

    def _define_action_space(self) -> List[Action]:
        """Enumerate the discrete actions used by the tabular agent."""
        actions = [Action("maintain")]

        for truck_id in range(1, self.num_trucks + 1):
            for target_slot in range(1, self.num_slots + 1):
                actions.append(Action("reschedule", truck_id, target_slot))

        for truck_id in range(1, self.num_trucks + 1):
            for incentive in [25, 50, 75]:
                actions.append(
                    Action(
                        "offer_incentive",
                        truck_id=truck_id,
                        incentive_amount=incentive,
                    )
                )

        for slot_id in range(1, self.num_slots + 1):
            for delta in [-1, 1]:
                actions.append(
                    Action(
                        "adjust_capacity",
                        target_slot=slot_id,
                        capacity_delta=delta,
                    )
                )

        return actions

    def get_state(self) -> SystemState:
        """Observe the current system, including a sampled time-period label."""
        current_time_period = 1 if random.random() < 0.6 else 2
        average_utilization = float(np.mean(list(self.slot_utilization.values())))
        congestion_level = (
            1 if average_utilization < 60
            else (2 if average_utilization < 80 else 3)
        )

        # A deep copy keeps this observation fixed when the environment transitions.
        return SystemState(
            truck_states=deepcopy(self.trucks),
            slot_utilization=self.slot_utilization.copy(),
            current_time_period=current_time_period,
            congestion_level=congestion_level,
        )

    def step(
        self, action: Action, state: SystemState
    ) -> Tuple[SystemState, float, bool]:
        """Score an action, apply its effects, and return the next observation."""
        reward = self._calculate_reward(action, state)
        self._apply_action(action)
        next_state = self.get_state()

        # This is a continuing process; fixed episode lengths are imposed by the agent.
        done = False
        return next_state, reward, done

    def _calculate_reward(self, action: Action, state: SystemState) -> float:
        """Combine simplified operating costs and action-specific reward effects."""
        reward = 0.0
        total_waiting_cost = sum(
            truck.expected_delay * self.waiting_cost_per_minute
            for truck in state.truck_states
        )
        total_operating_cost = self.num_trucks * self.operating_cost_per_slot
        reward -= total_operating_cost

        if action.type == "maintain":
            reward += 10.0

        elif action.type == "reschedule":
            reward -= self.rescheduling_cost
            truck = next(
                truck for truck in state.truck_states
                if truck.id == action.truck_id
            )
            # Slots 1-3 act as a simple proxy for earlier appointment opportunities.
            if truck.expected_delay > 30 and action.target_slot <= 3:
                reward += truck.expected_delay * self.waiting_cost_per_minute * 0.5

        elif action.type == "offer_incentive":
            reward -= float(action.incentive_amount)
            truck = next(
                truck for truck in state.truck_states
                if truck.id == action.truck_id
            )
            # Acceptance affects the reward only; no appointment change is simulated.
            if truck.urgency_level >= 2 and random.random() < 0.7:
                reward += truck.expected_delay * self.waiting_cost_per_minute * 0.3

        elif action.type == "adjust_capacity":
            reward -= abs(action.capacity_delta) * 20
            current_utilization = state.slot_utilization[action.target_slot]
            if action.capacity_delta > 0 and current_utilization > 80:
                reward += 50
            elif action.capacity_delta < 0 and current_utilization < 50:
                reward += 30

        reward -= total_waiting_cost * 0.1
        if state.congestion_level >= 3:
            reward -= 100

        return reward

    def _apply_action(self, action: Action) -> None:
        """Update the simplified appointment and utilization state."""
        if action.type == "reschedule":
            truck = next(
                truck for truck in self.trucks
                if truck.id == action.truck_id
            )
            truck.current_slot = action.target_slot
            slot = action.target_slot
            self.slot_utilization[slot] = min(
                100, self.slot_utilization[slot] + 10
            )

        elif action.type == "adjust_capacity":
            slot = action.target_slot
            change = action.capacity_delta * 10
            self.slot_utilization[slot] = int(
                np.clip(self.slot_utilization[slot] + change, 0, 100)
            )

        # Random delay changes make the transition process stochastic.
        if random.random() < 0.1:
            truck = random.choice(self.trucks)
            truck.expected_delay = max(
                0, truck.expected_delay + random.randint(-15, 15)
            )
            truck.urgency_level = (
                1 if truck.expected_delay < 20
                else (2 if truck.expected_delay < 40 else 3)
            )
            truck.location = (
                "on_time" if truck.expected_delay == 0
                else ("delayed" if truck.expected_delay < 45 else "very_delayed")
            )

## Q-Learning agent

The agent stores Q-values in a nested table indexed first by the encoded state and then by the action's index in the environment's fixed action list. For an unrecorded state-action pair, the value is treated as zero. During exploitation, ties among the best-valued actions are broken randomly so that action ordering does not determine every decision.

The update uses the largest next-state value across the complete action set. Since the environment does not produce terminal states, the update always includes this bootstrap term. The fixed number of steps per episode segment controls computation; it does not make the underlying process terminal.

In [ ]:
class QLearningAgent:
    """Tabular Q-Learning agent for truck appointment management."""

    def __init__(
        self,
        env: TruckAppointmentEnvironment,
        params: QLearningParameters,
    ) -> None:
        self.env = env
        self.params = params
        self.q_table = defaultdict(lambda: defaultdict(float))
        self.training_history = {
            "episode": [],
            "total_reward": [],
            "steps": [],
            "exploration_rate": [],
            "avg_reward_per_step": [],
        }

    def get_action_index(self, action: Action) -> int:
        """Return the stable index of an action in the environment action list."""
        return self.env.actions.index(action)

    def get_action_by_index(self, index: int) -> Action:
        """Retrieve an action by its stable index."""
        return self.env.actions[index]

    def choose_action(
        self, state_key: Tuple, exploration_rate: float
    ) -> Tuple[Action, int]:
        """Select an action with an epsilon-greedy policy."""
        if random.random() < exploration_rate:
            action_index = random.randrange(len(self.env.actions))
        else:
            q_values = self.q_table[state_key]
            values = [q_values.get(index, 0.0) for index in range(len(self.env.actions))]
            best_value = max(values)
            best_indices = [
                index for index, value in enumerate(values)
                if np.isclose(value, best_value)
            ]
            action_index = random.choice(best_indices)

        return self.get_action_by_index(action_index), action_index

    def update_q_value(
        self,
        state_key: Tuple,
        action_index: int,
        reward: float,
        next_state_key: Tuple,
        done: bool,
    ) -> None:
        """Apply the one-step Q-Learning update to a state-action estimate."""
        current_q = self.q_table[state_key][action_index]
        if done:
            max_next_q = 0.0
        else:
            next_values = self.q_table[next_state_key]
            max_next_q = max(
                0.0,
                max(next_values.values(), default=0.0),
            )

        target = reward + self.params.discount_factor * max_next_q
        self.q_table[state_key][action_index] = current_q + self.params.learning_rate * (
            target - current_q
        )

    def train_episode(self, episode_num: int) -> Dict[str, Any]:
        """Run one fixed-length segment of interaction with the environment."""
        state = self.env.get_state()
        state_key = state.get_state_key()
        total_reward = 0.0
        steps = 0

        exploration_rate = max(
            self.params.min_exploration_rate,
            self.params.exploration_rate
            * (self.params.exploration_decay ** episode_num),
        )

        for _ in range(self.params.max_steps_per_episode):
            action, action_index = self.choose_action(state_key, exploration_rate)
            next_state, reward, done = self.env.step(action, state)
            next_state_key = next_state.get_state_key()
            self.update_q_value(
                state_key, action_index, reward, next_state_key, done
            )

            state = next_state
            state_key = next_state_key
            total_reward += reward
            steps += 1

            if done:
                break

        return {
            "total_reward": total_reward,
            "steps": steps,
            "exploration_rate": exploration_rate,
            "avg_reward_per_step": total_reward / max(1, steps),
        }

    def train(self) -> Dict[str, Any]:
        """Train the agent and return a summary of the recorded training run."""
        print(f"Training for {self.params.num_episodes} episode segments...")
        start_time = time.time()

        for episode in range(self.params.num_episodes):
            result = self.train_episode(episode)
            self.training_history["episode"].append(episode + 1)
            self.training_history["total_reward"].append(result["total_reward"])
            self.training_history["steps"].append(result["steps"])
            self.training_history["exploration_rate"].append(
                result["exploration_rate"]
            )
            self.training_history["avg_reward_per_step"].append(
                result["avg_reward_per_step"]
            )

            if (episode + 1) % 1000 == 0:
                recent_average = np.mean(
                    self.training_history["total_reward"][-1000:]
                )
                print(
                    f"Segment {episode + 1}/{self.params.num_episodes}, "
                    f"mean reward over the last 1,000 segments: "
                    f"{recent_average:.2f}, "
                    f"exploration rate: {result['exploration_rate']:.3f}"
                )

        training_time = time.time() - start_time
        print(f"Training completed in {training_time:.2f} seconds.")

        return {
            "training_time": training_time,
            "final_exploration_rate": self.training_history["exploration_rate"][-1],
            "total_states_visited": len(self.q_table),
            "mean_reward_over_last_1000_segments": float(
                np.mean(self.training_history["total_reward"][-1000:])
            ),
        }

    def evaluate_policy(self, num_episodes: int = 100) -> Dict[str, Any]:
        """Evaluate the greedy policy in the stochastic environment."""
        if num_episodes <= 0:
            raise ValueError("num_episodes must be positive.")

        print(f"Evaluating the policy over {num_episodes} episode segments...")
        evaluation_rewards = []
        action_counts = defaultdict(int)

        for _ in range(num_episodes):
            state = self.env.get_state()
            state_key = state.get_state_key()
            episode_reward = 0.0

            for _ in range(self.params.max_steps_per_episode):
                action, _ = self.choose_action(state_key, 0.0)
                action_counts[action.type] += 1
                next_state, reward, done = self.env.step(action, state)
                episode_reward += reward
                state = next_state
                state_key = state.get_state_key()

                if done:
                    break

            evaluation_rewards.append(episode_reward)

        return {
            "mean_episode_reward": float(np.mean(evaluation_rewards)),
            "std_episode_reward": float(np.std(evaluation_rewards)),
            "min_episode_reward": float(np.min(evaluation_rewards)),
            "max_episode_reward": float(np.max(evaluation_rewards)),
            "action_distribution": dict(action_counts),
        }

## Create the environment and agent

The action list is constructed once for each environment, so an action index has a consistent meaning throughout training and evaluation. The environment and learning parameters below define the notebook's example run. The episode count and step limit are computational settings, not evidence that learning has converged.

In [ ]:
env = TruckAppointmentEnvironment(num_trucks=4, num_slots=6)
params = QLearningParameters(
    learning_rate=0.1,
    discount_factor=0.95,
    exploration_rate=1.0,
    exploration_decay=0.9995,
    min_exploration_rate=0.01,
    num_episodes=5000,
    max_steps_per_episode=50,
)

agent = QLearningAgent(env, params)

print(f"Environment: {env.num_trucks} trucks and {env.num_slots} slots")
print(f"Action-space size: {len(env.actions)} actions")
print(
    f"Learning rate: {params.learning_rate}, "
    f"discount factor: {params.discount_factor}, "
    f"training segments: {params.num_episodes}"
)

print("\nInitial truck states:")
for truck in env.trucks:
    print(
        f"Truck {truck.id}: slot {truck.current_slot}, "
        f"expected delay {truck.expected_delay} minutes, "
        f"urgency {truck.urgency_level}, location {truck.location}"
    )

## Train the agent

Each interaction supplies a reward and a next-state observation. The agent uses both in the Q-value update, allowing an action to be valued partly for the future states it tends to lead to. Training records the total reward and exploration rate for each fixed-length segment. Because the environment is stochastic and is not reset between segments, these records describe one evolving simulation run.

In [ ]:
training_results = agent.train()

print("\nTraining summary:")
for key, value in training_results.items():
    label = key.replace("_", " ").title()
    if isinstance(value, float):
        print(f"{label}: {value:.3f}")
    else:
        print(f"{label}: {value}")

## Evaluate the learned policy

Evaluation sets the exploration rate to zero, so the agent selects a currently highest-valued action, breaking ties at random. Random events in the environment still occur; therefore, evaluation rewards can vary across episode segments. Action counts describe decisions made during this evaluation run, not the frequency of actions in all possible terminal conditions.

In [ ]:
evaluation_results = agent.evaluate_policy(num_episodes=200)

print("\nPolicy evaluation summary:")
for key, value in evaluation_results.items():
    if key == "action_distribution":
        total_actions = sum(value.values())
        print("Action distribution:")
        for action_type, count in sorted(value.items()):
            percentage = 100 * count / total_actions if total_actions else 0.0
            print(f"  {action_type}: {count} ({percentage:.1f}%)")
    else:
        print(f"{key.replace('_', ' ').title()}: {value:.2f}")

## Visualize training and evaluation diagnostics

The reward plot includes both individual segment rewards and a moving average, which helps distinguish short-term randomness from a broader pattern in this run. The exploration plot shows the schedule used during training. The action chart summarizes evaluation decisions, and the heatmap displays Q-values for a small sample of recorded state-action pairs. These diagnostics are descriptive; by themselves, they do not establish convergence or policy quality.

In [ ]:
def create_learning_visualizations(
    agent: QLearningAgent,
    evaluation_results: Dict[str, Any],
) -> None:
    """Plot the recorded rewards, exploration schedule, actions, and Q-values."""
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(
        "Q-Learning Diagnostics for Truck Appointment Management",
        fontsize=16,
        fontweight="bold",
    )

    episodes = agent.training_history["episode"]
    rewards = agent.training_history["total_reward"]

    reward_axis = axes[0, 0]
    reward_axis.plot(
        episodes, rewards, alpha=0.3, color="tab:blue", label="Segment reward"
    )
    window_size = min(100, len(rewards))
    if window_size:
        moving_average = np.convolve(
            rewards, np.ones(window_size) / window_size, mode="valid"
        )
        moving_episodes = episodes[window_size - 1:]
        reward_axis.plot(
            moving_episodes,
            moving_average,
            color="tab:red",
            linewidth=2,
            label=f"Moving average ({window_size} segments)",
        )
    reward_axis.set_xlabel("Training segment")
    reward_axis.set_ylabel("Total reward (model score)")
    reward_axis.set_title("Training rewards")
    reward_axis.legend()
    reward_axis.grid(True, alpha=0.3)

    exploration_axis = axes[0, 1]
    exploration_axis.plot(
        episodes,
        agent.training_history["exploration_rate"],
        color="tab:purple",
        linewidth=2,
    )
    exploration_axis.axhline(
        y=agent.params.min_exploration_rate,
        color="tab:red",
        linestyle="--",
        label=f"Minimum exploration rate ({agent.params.min_exploration_rate})",
    )
    exploration_axis.set_xlabel("Training segment")
    exploration_axis.set_ylabel("Exploration rate, ε")
    exploration_axis.set_title("Exploration schedule")
    exploration_axis.legend()
    exploration_axis.grid(True, alpha=0.3)

    action_axis = axes[1, 0]
    action_distribution = evaluation_results["action_distribution"]
    action_types = list(action_distribution)
    counts = [action_distribution[action_type] for action_type in action_types]
    if action_types:
        bars = action_axis.bar(action_types, counts, color=sns.color_palette("husl", len(action_types)))
        total_actions = sum(counts)
        for bar, count in zip(bars, counts):
            percentage = 100 * count / total_actions if total_actions else 0.0
            action_axis.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height(),
                f"{percentage:.1f}%",
                ha="center",
                va="bottom",
            )
    action_axis.set_xlabel("Action type")
    action_axis.set_ylabel("Action count")
    action_axis.set_title("Actions during policy evaluation")
    action_axis.tick_params(axis="x", rotation=30)

    heatmap_axis = axes[1, 1]
    sample_states = list(agent.q_table.keys())[:10]
    sample_actions = list(range(min(20, len(agent.env.actions))))
    if sample_states and sample_actions:
        q_values = np.zeros((len(sample_states), len(sample_actions)))
        for row, state_key in enumerate(sample_states):
            for column, action_index in enumerate(sample_actions):
                q_values[row, column] = agent.q_table[state_key].get(
                    action_index, 0.0
                )
        image = heatmap_axis.imshow(
            q_values, cmap="viridis", aspect="auto"
        )
        heatmap_axis.set_xlabel("Action index in the environment action list")
        heatmap_axis.set_ylabel("Sampled state index")
        heatmap_axis.set_title("Q-values for sampled states and actions")
        colorbar = fig.colorbar(image, ax=heatmap_axis)
        colorbar.set_label("Q-value (model score)")
    else:
        heatmap_axis.text(
            0.5, 0.5, "No state-action values are available to display.",
            ha="center", va="center", transform=heatmap_axis.transAxes,
        )
        heatmap_axis.set_title("Q-values for sampled states and actions")

    fig.tight_layout()
    plt.show()


create_learning_visualizations(agent, evaluation_results)

## Inspect the learned Q-table

For each recorded state, the largest stored Q-value identifies a greedy action according to the current table. Grouping state keys by whether any truck exceeds the delay threshold gives a coarse descriptive view of the table. The Q-values are estimates learned from this simulation; they are not guaranteed optimal values. A displayed action type can represent multiple distinct actions, such as rescheduling different trucks or targeting different slots.

In [ ]:
def analyze_learned_policy(agent: QLearningAgent) -> None:
    """Summarize greedy action preferences and recorded Q-value statistics."""
    print("Learned Q-table analysis")
    state_action_preferences = defaultdict(lambda: defaultdict(int))

    for state_key, q_values in agent.q_table.items():
        if not q_values:
            continue

        values = [
            q_values.get(index, 0.0)
            for index in range(len(agent.env.actions))
        ]
        best_value = max(values)
        best_indices = [
            index for index, value in enumerate(values)
            if np.isclose(value, best_value)
        ]
        delay_pattern, _, _, _ = state_key
        state_type = "At least one delay above 30 minutes" if any(
            delay_pattern
        ) else "No delay above 30 minutes"

        # Count one greedy action per state; random tie-breaking avoids index bias.
        best_action = agent.get_action_by_index(random.choice(best_indices))
        state_action_preferences[state_type][best_action.type] += 1

    if state_action_preferences:
        print("\nGreedy action types by coarse state group:")
        for state_type, action_counts in state_action_preferences.items():
            total = sum(action_counts.values())
            print(f"\n{state_type}:")
            for action_type, count in sorted(
                action_counts.items(), key=lambda item: item[1], reverse=True
            ):
                print(f"  {action_type}: {count} ({100 * count / total:.1f}%)")
    else:
        print("No Q-table states are available for analysis.")

    all_q_values = [
        value
        for state_values in agent.q_table.values()
        for value in state_values.values()
    ]
    if all_q_values:
        print("\nRecorded Q-value statistics:")
        print(f"  Number of stored values: {len(all_q_values)}")
        print(f"  Mean: {np.mean(all_q_values):.2f}")
        print(f"  Standard deviation: {np.std(all_q_values):.2f}")
        print(f"  Minimum: {np.min(all_q_values):.2f}")
        print(f"  Maximum: {np.max(all_q_values):.2f}")

        print("\nSample state entries and highest-valued actions:")
        for state_number, (state_key, q_values) in enumerate(
            list(agent.q_table.items())[:5], start=1
        ):
            print(f"\nState {state_number}: {state_key}")
            ranked_actions = sorted(
                q_values.items(), key=lambda item: item[1], reverse=True
            )
            for action_index, q_value in ranked_actions[:3]:
                action = agent.get_action_by_index(action_index)
                print(
                    f"  {action.type} (action index {action_index}): "
                    f"{q_value:.2f}"
                )
    else:
        print("No state-action values are available for summary statistics.")


analyze_learned_policy(agent)

## Comparing measured methods

A reward from this simulation cannot be converted into monetary cost or waiting time without a validated relationship between the reward components and those operational measures. Baseline figures must also come from actual measurements under comparable scenarios; illustrative or invented figures would not support a valid comparison.

The function below is an optional plotting utility for a dataset containing measured average cost and average waiting time for each method. It does not create baseline data or estimate these quantities from Q-Learning rewards. Supply such measurements only when they are available and use consistent units and evaluation conditions.

In [ ]:
def compare_with_baseline_methods(
    measured_results: Dict[str, Dict[str, float]],
) -> None:
    """Plot supplied measured costs and waiting times without inventing data.

    Each method entry must contain ``average_cost`` and
    ``average_waiting_time`` measured under comparable evaluation conditions.
    """
    if not measured_results:
        raise ValueError("Provide measured results for at least one method.")

    required_metrics = {"average_cost", "average_waiting_time"}
    for method, metrics in measured_results.items():
        missing = required_metrics.difference(metrics)
        if missing:
            raise ValueError(
                f"{method} is missing measured metrics: {sorted(missing)}"
            )

    methods = list(measured_results)
    costs = [measured_results[method]["average_cost"] for method in methods]
    waiting_times = [
        measured_results[method]["average_waiting_time"] for method in methods
    ]

    figure, (cost_axis, waiting_axis) = plt.subplots(
        1, 2, figsize=(14, 5)
    )
    cost_axis.bar(methods, costs, color="tab:blue")
    cost_axis.set_ylabel("Measured average cost (units supplied by the data)")
    cost_axis.set_title("Measured cost by method")
    cost_axis.tick_params(axis="x", rotation=30)

    waiting_axis.bar(methods, waiting_times, color="tab:orange")
    waiting_axis.set_ylabel("Measured average waiting time (minutes)")
    waiting_axis.set_title("Measured waiting time by method")
    waiting_axis.tick_params(axis="x", rotation=30)

    figure.tight_layout()
    plt.show()

## Results interpretation and limitations

Interpret the outputs as diagnostics of the stated simulation and Q-Learning implementation:

- **Training rewards:** A moving average can make broad changes easier to see, but reward variation is expected in a stochastic environment. An upward trend alone does not establish convergence or optimality.
- **Exploration rate:** The plotted schedule shows how often the agent is allowed to explore. The decay parameter controls the rate of decline, while the minimum exploration rate prevents exploration from reaching zero during training.
- **Evaluation rewards:** The mean, standard deviation, minimum, and maximum summarize outcomes over the sampled evaluation segments. They depend on the simulation's random events and current environment state.
- **Action counts:** These counts describe the greedy policy's decisions in the observed evaluation run. They do not show that an action is best in every state or that it will perform similarly in a real terminal.
- **Q-table summaries:** The table covers encountered, aggregated states. States not encountered during training have no learned experience; unseen action values are treated as zero when selecting a greedy action.
- **Method comparisons:** A fair comparison requires independently measured methods, the same scenarios, matching cost and service definitions, and an evaluation protocol that accounts for stochastic variation. The simulation reward alone is not a measured cost or waiting-time result.

The state representation, reward design, and transition rules determine what the agent can learn. In particular, the simplified incentive and utilization dynamics limit the operational meaning of the resulting policy. Larger or more realistic studies could extend the environment with additional state information, appointment acceptance and service outcomes, calibrated cost data, or additional operational constraints. Such extensions require explicit modeling and validation; they are outside the scope of this example.